# LD-O1 — diagnostic des lignes de bord manquées (CPU, mf_0045-0049, LD-2 gelé)

In [ ]:

import time
T_START = time.monotonic()          # pris au tout début de la cellule
import datetime, faulthandler, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld_o1 start', _utc(), flush=True)
OUT = Path('/kaggle/working')
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld-o1-kit', '/kaggle/input/datasets/mabaku/babaorum-ld-o1-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
LD2_CANDIDATES = ['/kaggle/input/babaorum-ld2-run/ld2', '/kaggle/input/notebooks/mabaku/babaorum-ld2-run/ld2']
LD2_SHA = 'e5afd1432e10aa8e06191678844ae350348f7586f505d1a766aba3a5a44b42c3'
LD2_FILE = 'ld2_main_step003271.pt'
PAGE_IDS = ['mf_0045', 'mf_0046', 'mf_0047', 'mf_0048', 'mf_0049']
TIMEOUTS = {'D': 900, 'RUN1': 1800, 'RUN4': 5400, 'REPORT': 300}
LIMITS = {'deadline_s': 7200, 'stop_margin_s': 120, 'heartbeat_s': 60}
RUN_FILES = ['scripts/diag_ld_o1.py', 'config/ld_o1.json', 'tests/test_ld_o1.py']
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld_o1.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
LD2DIR = resolve_one(LD2_CANDIDATES, "ld2_manifest.json")
assert sha256(LD2DIR / LD2_FILE) == LD2_SHA, "checkpoint LD-2 : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)
import ld2_core as C2

RUN = OUT / "ld_o1"
EXITS = OUT / "exits"
class Incomplete(Exception):
    pass

def remaining():
    return LIMITS["deadline_s"] - (time.monotonic() - T_START)

def allowed(cap, reserve=0.0):
    """max(0, min(plafond, restant - réserve))."""
    return C2.allowed(cap, remaining(), reserve)

def step_argv(ph, build_argv, cap, reserve=0.0, **kw):
    def argv():
        t = allowed(cap, reserve)
        if t <= 0:
            raise Incomplete(f"échéance globale atteinte avant l'étape {ph}")
        st["timeout_s"] = t
        return build_argv()
    st = {"name": ph, "argv": argv, "timeout_s": cap, **kw}
    return st

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "diag_ld_o1.py"), "--phase", phase, "--root", str(PAGES), "--out", str(RUN),
            "--ld2-dir", str(LD2DIR), "--cache-dir", "/tmp/ld_o1_hf", *extra]

def guard_fn(name, fn):
    def run():
        if remaining() - LIMITS["stop_margin_s"] <= 0:
            raise Incomplete(f"échéance globale atteinte avant l'étape {name}")
        return fn()
    return {"name": name, "fn": run}

def ld_o1_cleanup(rec):
    """TIMEOUT, échéance ou échec d'étape : INCOMPLETE à l'étape, sauf si l'étape a écrit FAILED (refus, garde)."""
    if rec["status"] == "TIMEOUT" or str(rec.get("exception") or "").startswith("Incomplete:"):
        rec["status"] = "INCOMPLETE"
    elif rec["status"] == "FAILED" and rec["exit_code"] != "EXCEPTION":
        f = EXITS / f"ld_o1_run_exit_{rec['step']}.json"
        if not f.is_file() or json.loads(f.read_text()).get("status") != "FAILED":
            rec["status"] = "INCOMPLETE"

def s_pre():
    import torch
    (OUT / "ld_o1_env.json").write_text(json.dumps({"python": sys.version, "torch": torch.__version__}, indent=1))
    print("pre ok", _utc(), flush=True)

# TOUTE étape argv est bornée par l'échéance globale (7 200 s) moins la marge d'arrêt (120 s : grâce du kill d'execute_pipeline,
# SIGTERM puis SIGKILL après 10 s + 5 s d'attente, et nettoyage final).
M = LIMITS["stop_margin_s"]
OFFLINE = lambda: dict(os.environ, HF_HUB_OFFLINE="1")       # hors ligne sauf D
D_STEP = step_argv("D", lambda: _argv("D"), TIMEOUTS["D"], M)
RUN1 = step_argv("RUN1", lambda: _argv("RUN", ["--pages", PAGE_IDS[0]]), TIMEOUTS["RUN1"], M, env=OFFLINE)    # fumée : mf_0045 SEULE
RUN4 = step_argv("RUN4", lambda: _argv("RUN", ["--pages", *PAGE_IDS[1:]]), TIMEOUTS["RUN4"], M, env=OFFLINE)
REPORT = step_argv("REPORT", lambda: _argv("REPORT"), TIMEOUTS["REPORT"], M, env=OFFLINE)
STEPS = [guard_fn("pre", s_pre), D_STEP, RUN1, RUN4, REPORT]
EXIT = None
try:
    EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=LIMITS["heartbeat_s"], cleanup=ld_o1_cleanup)
finally:
    _rec = dict(EXIT) if EXIT is not None else {"status": "FAILED_BEFORE_EXIT"}
    _rec["kit_sha256"] = {rel: sha256(KIT / rel) for rel in RUN_FILES if (KIT / rel).is_file()}
    (OUT / "ld_o1_run_exit.json").write_text(json.dumps(_rec, indent=1, default=str))
print("EXIT", json.dumps({k: EXIT[k] for k in ("status", "step", "exit_code")}) if EXIT else "none", flush=True)
if (RUN / "ld_o1_report.json").is_file():
    print("REPORT_SHA256", sha256(RUN / "ld_o1_report.json"), flush=True)
